## Overview 
Your team at EngageMetrics, a leading employee engagement analytics company, has just received datasets from multiple sources that need to be consolidated for an urgent executive presentation. 
The data includes employee feedback from regional offices in CSV format, educational achievement records in Excel spreadsheets, and compensation benchmarks from an external API. The leadership team needs insights on employee engagement trends by tomorrow morning.
<br>
The goal is to efficiently import and process this scattered data using Python. 
EngageMetrics datasets containing employee insights, educational records, and external market data, are used to work with the essential data import techniques needed to handle various file formats and common data challenges.

## Learning Outcomes 

- Import data from CSV files using pandas
- Load data from Excel files with proper formatting
- Connect to and retrieve data from a REST API
- Handle common data import challenges (missing values, inconsistent formats)

## Dataset Information 
Two different EngageMetrics data sources and an external API from Wikipedia:

1. <b>employee_insights.csv:</b> Employee performance and demographic data
2. <b>education_data.xlsx:</b> Educational background information
3. <b>Wikipedia REST API:</b> Income statistics in the United States

<b>1. imports</b>

In [ ]:
import pandas as pd
import requests
import json
from datetime import datetime
from pathlib import Path

<b>2. path adjustments</b>

In [ ]:

BASE_DIR = Path(__file__).resolve().parents[2]
DATA_DIR = BASE_DIR / "data"
employee_insights_path = DATA_DIR / "employee_insights.csv"
education_data_path = DATA_DIR / "education_data.xlsx"

<b>3. utils functions</b>

In [ ]:
def convert_dates(df: pd.DataFrame, date_cols: list[str]) -> pd.DataFrame:
    valid_df = df.copy()
    for col in date_cols:
        valid_df[col] = pd.to_datetime(df[col], errors='coerce')
        invalid_mask = valid_df[col].isna() & df[col].notna()
        invalid_values = df.loc[invalid_mask, col].sample(n=2)
        if len(invalid_values) > 0:
            print(f'invalid values found at column {col}')
            print(invalid_values)
            raise ValueError()

    return valid_df


def validate_year(value) -> bool:
    try:
        year = int(value)
        return 1900 <= year <= datetime.now().year
    except ValueError:
        return False

## Activities

### Activity 1: Regional Feedback Data Import

Import employee insights data from a CSV file. Import the quarterly employee engagement feedback data from all regions.

<b>Step 1:</b> Load data

In [ ]:
employee_df = pd.read_csv(employee_insights_path)
employee_df.head(5)

<b>Step 2:</b> Check data quality

In [ ]:
employee_df.info()

In [ ]:
employee_df.isna().sum()

In [ ]:
# Identify inconsistent date formats 
date_cols = ['last_promotion_date', 'last_training_date']
employee_df = convert_dates(df=employee_df, date_cols=date_cols)

### Activity 2: Educational Records Import
Import educational background data from an Excel file. Process the talent development team's educational background data.

<b>Step 1:</b> Load data

In [ ]:
education_df = pd.read_excel(education_data_path)
education_df.head(5)

<b>Step 2:</b> Check data quality

In [ ]:
education_df.info()

In [ ]:
education_df.isna().sum()

In [ ]:
# Verify whether graduation years are in the correct format
invalid_years = education_df[~education_df['graduation_year'].apply(validate_year)]
print('invalid years found:')
print(invalid_years)

In [ ]:
# Check if there are any missing educational backgrounds
education_df[education_df["educational_background"].isna()]

### Activity 3: Compensation Benchmark Data Import
Retrieve data from the Wikipedia API. Retrieve industry compensation data for comparison.

<b>Step 1:</b> Make an API request

In [ ]:
url = "https://en.wikipedia.org/api/rest_v1/page/summary/Income_in_the_United_States"

try:
    response = requests.get(url)
except requests.RequestException as e:
    print(f'error:{e}')
    exit(1)

# Parse the JSON response
if response.status_code == 200:
    data = response.json()
else:
    print(f'response error status: {response.status_code}')
    print(response)
# YOUR CODE HERE 